In [ ]:
# Cell 1: Secrets
import os
os.environ["HF_TOKEN"] = "..."
os.environ["COMET_API_KEY"] = "..."
os.environ["HF_MODEL_REPO"] = "sk3feel/docvqa-privacy-checkpoints"


In [ ]:
# Cell 2: Setup
!git clone https://github.com/sk3feel/hidden-data-reproduction-multimodal.git /content/course_work2026 2>/dev/null || (cd /content/course_work2026 && git pull)
%cd /content/course_work2026
!pip install -q -r requirements.txt
!pip install -q transformers accelerate Pillow pandas tqdm einops timm comet_ml huggingface_hub
!pip install -q peft bitsandbytes trl qwen-vl-utils
!python src/download_from_hf.py --repo-id sk3feel/docvqa-privacy-data

from huggingface_hub import snapshot_download
snapshot_download("sk3feel/docvqa-privacy-checkpoints", local_dir="checkpoints/", repo_type="model", token=os.environ["HF_TOKEN"])


In [ ]:
# Cell 3: Utilities
import gc, json, os, re, torch, pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageFile
from transformers import AutoModelForCausalLM, AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import roc_auc_score
from scipy.stats import ttest_ind
from tqdm.auto import tqdm
from qwen_vl_utils import process_vision_info
import matplotlib.pyplot as plt
import seaborn as sns
import comet_ml

ImageFile.LOAD_TRUNCATED_IMAGES = True
MAX_IMAGE_SIDE = 1024
MIA_DIR = Path("artifacts/privacy_attacks/mia")
MIA_DIR.mkdir(parents=True, exist_ok=True)


def save_csv_with_comet(df, path, experiment, table_name=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    csv_string = df.to_csv(index=False)
    experiment.log_asset(str(path), file_name=path.name)
    experiment.log_asset_data(csv_string, file_name=path.name)
    if table_name is not None:
        experiment.log_table(table_name, df)
    return path


def safe_open_image(path, max_side=MAX_IMAGE_SIDE):
    image = Image.open(path).convert("RGB")
    if max(image.size) > max_side:
        image.thumbnail((max_side, max_side), Image.LANCZOS)
    return image


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def enrich_with_field_types(records, manifest_path):
    manifest = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                manifest.append(json.loads(line))
    type_map = {}
    for row in manifest:
        eid = str(row.get("example_id", row.get("question_id", "")))
        type_map[eid] = row.get("coarse_field_type", "OTHER")
    for rec in records:
        rec["coarse_field_type"] = type_map.get(rec["example_id"], "OTHER")
    return records


def stratified_sample(records, n, type_key="coarse_field_type"):
    by_type = defaultdict(list)
    for rec in records:
        by_type[rec[type_key]].append(rec)
    types = sorted(by_type.keys())
    per_type = n // len(types)
    result = []
    for ft in types:
        result.extend(by_type[ft][:per_type])
    used = {r["example_id"] for r in result}
    for rec in records:
        if len(result) >= n:
            break
        if rec["example_id"] not in used:
            result.append(rec)
    return result[:n]


def make_qwen_user_text(question):
    return f"Answer the question about this document image. Give only the answer value, nothing else.\nQuestion: {question}"


def compute_confidence_and_loss(model, processor, image, question, gold_answer, model_type="florence2"):
    model.eval()
    with torch.no_grad():
        if model_type == "florence2":
            inputs = processor(text=f"<VQA>{question}", images=image, return_tensors="pt").to(model.device)
            labels = processor.tokenizer(gold_answer, return_tensors="pt").input_ids.to(model.device)
            outputs = model(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], labels=labels)
            loss = outputs.loss.item()
            generate_inputs = {"input_ids": inputs["input_ids"], "pixel_values": inputs["pixel_values"]}
            gen = model.generate(**generate_inputs, max_new_tokens=50, output_scores=True, return_dict_in_generate=True)
            log_probs = []
            prompt_len = inputs["input_ids"].shape[1]
            for i, scores in enumerate(gen.scores):
                lsm = torch.nn.functional.log_softmax(scores[0], dim=-1)
                tok = gen.sequences[0, i + prompt_len]
                if tok == processor.tokenizer.eos_token_id:
                    break
                log_probs.append(lsm[tok].item())
            confidence = sum(log_probs) / max(len(log_probs), 1)
        elif model_type == "qwen":
            messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": make_qwen_user_text(question)}]}]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            img_inputs, _ = process_vision_info(messages)
            img_input = img_inputs[0] if isinstance(img_inputs, list) else img_inputs
            inputs = processor(text=[text], images=[img_input], return_tensors="pt", padding=True)
            device = next(model.parameters()).device
            inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
            full_messages = messages + [{"role": "assistant", "content": [{"type": "text", "text": gold_answer}]}]
            full_text = processor.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
            full_inputs = processor(text=[full_text], images=[img_input], return_tensors="pt", padding=True)
            full_inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in full_inputs.items()}
            labels = full_inputs["input_ids"].clone()
            labels[full_inputs["attention_mask"] == 0] = -100
            labels[:, :inputs["input_ids"].shape[1]] = -100
            outputs = model(**full_inputs, labels=labels)
            loss = outputs.loss.item()
            gen = model.generate(**inputs, max_new_tokens=50, output_scores=True, return_dict_in_generate=True)
            log_probs = []
            prompt_len = inputs["input_ids"].shape[1]
            for i, scores in enumerate(gen.scores):
                lsm = torch.nn.functional.log_softmax(scores[0], dim=-1)
                tok = gen.sequences[0, i + prompt_len]
                if tok == processor.tokenizer.eos_token_id:
                    break
                log_probs.append(lsm[tok].item())
            confidence = sum(log_probs) / max(len(log_probs), 1)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")
    return confidence, loss


def run_mia(model, processor, seen, unseen, model_type):
    results = []
    for rec in tqdm(seen, desc="MIA seen"):
        img = safe_open_image(rec["image_path"])
        c, l = compute_confidence_and_loss(model, processor, img, rec["question"], rec["answer"], model_type)
        results.append({"example_id": rec["example_id"], "split": "seen", "field_type": rec["coarse_field_type"], "confidence": c, "loss": l})
    for rec in tqdm(unseen, desc="MIA unseen"):
        img = safe_open_image(rec["image_path"])
        c, l = compute_confidence_and_loss(model, processor, img, rec["question"], rec["answer"], model_type)
        results.append({"example_id": rec["example_id"], "split": "unseen", "field_type": rec["coarse_field_type"], "confidence": c, "loss": l})
    return pd.DataFrame(results)


def compute_mia_metrics(df):
    labels = (df["split"] == "seen").astype(int)
    sc = df[df["split"] == "seen"]["confidence"]
    uc = df[df["split"] == "unseen"]["confidence"]
    sl = df[df["split"] == "seen"]["loss"]
    ul = df[df["split"] == "unseen"]["loss"]
    t_stat, p_value = ttest_ind(sc, uc, equal_var=False)
    return {
        "auc_confidence": roc_auc_score(labels, df["confidence"]),
        "auc_loss": roc_auc_score(labels, -df["loss"]),
        "t_stat": t_stat,
        "p_value": p_value,
        "mean_seen_conf": sc.mean(),
        "std_seen_conf": sc.std(),
        "mean_unseen_conf": uc.mean(),
        "std_unseen_conf": uc.std(),
        "mean_seen_loss": sl.mean(),
        "mean_unseen_loss": ul.mean(),
    }


def plot_histogram(df, title):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df[df["split"] == "seen"]["confidence"], bins=30, alpha=0.6, label="seen", color="red")
    ax.hist(df[df["split"] == "unseen"]["confidence"], bins=30, alpha=0.6, label="unseen", color="blue")
    ax.set_xlabel("??????????? ?????? (??????? log-prob)")
    ax.set_ylabel("?????????? ????????")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    return fig


def cleanup_gpu():
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# Cell 4: Data preparation
qwen_train_ids_path = Path("artifacts/finetuning_generative/qwen_train_200_ids.csv")
assert qwen_train_ids_path.exists(), "Missing qwen_train_200_ids.csv from notebook 21"
qwen_train_ids = pd.read_csv(qwen_train_ids_path)["example_id"].astype(str).tolist()

all_train = load_jsonl("artifacts/docqa_recovery/benchmark_train/manifest.jsonl")
all_train_map = {str(r.get("example_id", r.get("question_id", ""))): r for r in all_train}

qwen_seen = []
for eid in qwen_train_ids:
    if eid in all_train_map:
        r = all_train_map[eid]
        image_name = r.get("image_path", "").replace("\\", "/").split("/")[-1]
        answer_value = r["answers"][0] if isinstance(r["answers"], list) else r["answers"]
        qwen_seen.append({
            "example_id": eid,
            "image_path": f"artifacts/docqa_recovery/benchmark_train/images/original/{image_name}",
            "question": r["question"],
            "answer": answer_value,
            "coarse_field_type": r.get("coarse_field_type", "OTHER"),
        })

florence_all_train = []
for r in all_train:
    eid = str(r.get("example_id", r.get("question_id", "")))
    image_name = r.get("image_path", "").replace("\\", "/").split("/")[-1]
    answer_value = r["answers"][0] if isinstance(r["answers"], list) else r["answers"]
    florence_all_train.append({
        "example_id": eid,
        "image_path": f"artifacts/docqa_recovery/benchmark_train/images/original/{image_name}",
        "question": r["question"],
        "answer": answer_value,
        "coarse_field_type": r.get("coarse_field_type", "OTHER"),
    })
florence_seen = stratified_sample(florence_all_train, 200)

all_val = load_jsonl("artifacts/docqa_recovery/benchmark/manifest.jsonl")
val_records = []
for r in all_val:
    eid = str(r.get("example_id", r.get("question_id", "")))
    image_name = r.get("image_path", "").replace("\\", "/").split("/")[-1]
    answer_value = r["answers"][0] if isinstance(r["answers"], list) else r["answers"]
    val_records.append({
        "example_id": eid,
        "image_path": f"artifacts/docqa_recovery/benchmark/images/original/{image_name}",
        "question": r["question"],
        "answer": answer_value,
        "coarse_field_type": r.get("coarse_field_type", "OTHER"),
    })
unseen_200 = stratified_sample(val_records, 200)

print(f"Florence seen: {len(florence_seen)}, Qwen seen: {len(qwen_seen)}, Unseen: {len(unseen_200)}")


In [ ]:
# Cell 5: Baseline MIA
os.makedirs("artifacts/privacy_attacks/mia", exist_ok=True)
experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
experiment.set_name("mia-baseline")

baseline_summary_rows = []

model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-base", trust_remote_code=True).to("cuda")
proc = AutoProcessor.from_pretrained("microsoft/Florence-2-base", trust_remote_code=True)
df = run_mia(model, proc, florence_seen, unseen_200, "florence2")
m = compute_mia_metrics(df)
print(f"Florence baseline: AUC_conf={m['auc_confidence']:.3f}")
save_csv_with_comet(df, MIA_DIR / "baseline_florence2.csv", experiment)
experiment.log_metrics({f"baseline_florence2_{k}": v for k, v in m.items()})
baseline_summary_rows.append({"tag": "florence2", **m, "n_seen": len(florence_seen), "n_unseen": len(unseen_200)})
del model, proc
cleanup_gpu()

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
model = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", quantization_config=bnb, device_map="auto", trust_remote_code=True)
model.eval()
proc = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", trust_remote_code=True)
df = run_mia(model, proc, qwen_seen, unseen_200, "qwen")
m = compute_mia_metrics(df)
print(f"Qwen-2B baseline: AUC_conf={m['auc_confidence']:.3f}")
save_csv_with_comet(df, MIA_DIR / "baseline_qwen2b.csv", experiment)
experiment.log_metrics({f"baseline_qwen2b_{k}": v for k, v in m.items()})
baseline_summary_rows.append({"tag": "qwen2b", **m, "n_seen": len(qwen_seen), "n_unseen": len(unseen_200)})
del model, proc
cleanup_gpu()

baseline_summary_df = pd.DataFrame(baseline_summary_rows)
save_csv_with_comet(baseline_summary_df, MIA_DIR / "baseline_mia_summary.csv", experiment, table_name="baseline_mia_summary.csv")
display(baseline_summary_df)

experiment.end()


In [ ]:
# Cell 6: Fine-tuned MIA
experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
experiment.set_name("mia-finetuned")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
finetuned_summary_rows = []

model = AutoModelForCausalLM.from_pretrained("checkpoints/florence2/epoch_30", trust_remote_code=True).to("cuda")
proc = AutoProcessor.from_pretrained("checkpoints/florence2/epoch_30", trust_remote_code=True)
df = run_mia(model, proc, florence_seen, unseen_200, "florence2")
m = compute_mia_metrics(df)
print(f"Florence ft: AUC_conf={m['auc_confidence']:.3f}, p={m['p_value']:.4f}")
save_csv_with_comet(df, MIA_DIR / "florence2_epoch30.csv", experiment)
fig = plot_histogram(df, "Florence-2 (????? 30)")
experiment.log_figure("florence2_hist", fig)
plt.close(fig)
experiment.log_metrics({f"florence2_epoch30_{k}": v for k, v in m.items()})
finetuned_summary_rows.append({"tag": "florence2", "checkpoint": "epoch_30", **m, "n_seen": len(florence_seen), "n_unseen": len(unseen_200)})
del model, proc
cleanup_gpu()

base = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", quantization_config=bnb, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, "checkpoints/qwen2b/epoch_5")
model.eval()
proc = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct", trust_remote_code=True)
df = run_mia(model, proc, qwen_seen, unseen_200, "qwen")
m = compute_mia_metrics(df)
print(f"Qwen-2B ft: AUC_conf={m['auc_confidence']:.3f}, p={m['p_value']:.4f}")
save_csv_with_comet(df, MIA_DIR / "qwen2b_epoch5.csv", experiment)
fig = plot_histogram(df, "Qwen2-VL-2B (????? 5)")
experiment.log_figure("qwen2b_hist", fig)
plt.close(fig)
experiment.log_metrics({f"qwen2b_epoch5_{k}": v for k, v in m.items()})
finetuned_summary_rows.append({"tag": "qwen2b", "checkpoint": "epoch_5", **m, "n_seen": len(qwen_seen), "n_unseen": len(unseen_200)})
del model, base, proc
cleanup_gpu()

base = Qwen2VLForConditionalGeneration.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", quantization_config=bnb, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, "checkpoints/qwen7b/epoch_5")
model.eval()
proc = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", trust_remote_code=True)
df = run_mia(model, proc, qwen_seen, unseen_200, "qwen")
m = compute_mia_metrics(df)
print(f"Qwen-7B ft: AUC_conf={m['auc_confidence']:.3f}, p={m['p_value']:.4f}")
save_csv_with_comet(df, MIA_DIR / "qwen7b_epoch5.csv", experiment)
fig = plot_histogram(df, "Qwen2-VL-7B (????? 5)")
experiment.log_figure("qwen7b_hist", fig)
plt.close(fig)
experiment.log_metrics({f"qwen7b_epoch5_{k}": v for k, v in m.items()})
finetuned_summary_rows.append({"tag": "qwen7b", "checkpoint": "epoch_5", **m, "n_seen": len(qwen_seen), "n_unseen": len(unseen_200)})
del model, base, proc
cleanup_gpu()

finetuned_summary_df = pd.DataFrame(finetuned_summary_rows)
save_csv_with_comet(finetuned_summary_df, MIA_DIR / "finetuned_mia_summary.csv", experiment, table_name="finetuned_mia_summary.csv")
display(finetuned_summary_df)

experiment.end()


In [ ]:
# Cell 7: AUC by field type
summary_experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
summary_experiment.set_name("mia-summary")

baseline_summary_df = pd.read_csv(MIA_DIR / "baseline_mia_summary.csv")
finetuned_summary_df = pd.read_csv(MIA_DIR / "finetuned_mia_summary.csv")
summary_experiment.log_table("baseline_mia_summary.csv", baseline_summary_df)
summary_experiment.log_table("finetuned_mia_summary.csv", finetuned_summary_df)

fine_tuned_files = {
    "florence2_epoch30": MIA_DIR / "florence2_epoch30.csv",
    "qwen2b_epoch5": MIA_DIR / "qwen2b_epoch5.csv",
    "qwen7b_epoch5": MIA_DIR / "qwen7b_epoch5.csv",
}
rows = []
for model_tag, csv_path in fine_tuned_files.items():
    df = pd.read_csv(csv_path)
    for field_type, group in df.groupby("field_type"):
        labels = (group["split"] == "seen").astype(int)
        if labels.nunique() < 2:
            continue
        rows.append({
            "model": model_tag,
            "field_type": field_type,
            "auc_confidence": roc_auc_score(labels, group["confidence"]),
            "auc_loss": roc_auc_score(labels, -group["loss"]),
            "n_examples": len(group),
        })
field_auc_df = pd.DataFrame(rows).sort_values(["model", "field_type"]).reset_index(drop=True)
save_csv_with_comet(field_auc_df, MIA_DIR / "field_type_auc.csv", summary_experiment, table_name="field_type_auc.csv")

fig, ax = plt.subplots(figsize=(10, 4))
heatmap_df = field_auc_df.pivot(index="model", columns="field_type", values="auc_confidence")
sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
ax.set_title("MIA AUC-ROC ?? ????? ?????")
ax.set_xlabel("??? ????")
ax.set_ylabel("??????")
plt.tight_layout()
summary_experiment.log_figure("field_type_auc_heatmap", fig)
plt.close(fig)

mia_bar_rows = []
for _, row in baseline_summary_df.iterrows():
    mia_bar_rows.append({"tag": row["tag"], "stage": "baseline", "auc_confidence": row["auc_confidence"], "auc_loss": row["auc_loss"]})
for _, row in finetuned_summary_df.iterrows():
    mia_bar_rows.append({"tag": row["tag"], "stage": "fine-tuned", "auc_confidence": row["auc_confidence"], "auc_loss": row["auc_loss"]})
mia_bar_df = pd.DataFrame(mia_bar_rows)
save_csv_with_comet(mia_bar_df, MIA_DIR / "mia_auc_bar_summary.csv", summary_experiment, table_name="mia_auc_bar_summary.csv")

plot_df = mia_bar_df.melt(id_vars=["tag", "stage"], value_vars=["auc_confidence", "auc_loss"], var_name="metric", value_name="auc")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, metric in zip(axes, ["auc_confidence", "auc_loss"]):
    metric_df = plot_df[plot_df["metric"] == metric]
    sns.barplot(data=metric_df, x="tag", y="auc", hue="stage", ax=ax, palette="Set2")
    ax.set_title("AUC-ROC ?? " + ("confidence" if metric == "auc_confidence" else "loss"))
    ax.set_xlabel("??????")
    ax.set_ylabel("AUC-ROC")
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)
plt.tight_layout()
summary_experiment.log_figure("mia_auc_bar_baseline_vs_finetuned", fig)
plt.close(fig)

display(field_auc_df)
display(mia_bar_df)
summary_experiment.end()


In [ ]:
# Cell 8: Saved outputs
saved_csvs = sorted(str(path) for path in MIA_DIR.glob("*.csv"))
display(pd.DataFrame({"csv_path": saved_csvs}))
print({"csv_count": len(saved_csvs), "output_dir": str(MIA_DIR)})
